In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql.functions import col

# Carga del dataset

transactions = spark.table("ml_layer.transaction_features_clean")

# Data split with stratification to avoid leakage on upcoming fit

fraud = transactions.filter("is_fraud = 1")
legit = transactions.filter("is_fraud = 0")

train_fraud, test_fraud = fraud.randomSplit([0.8, 0.2], seed=42)
train_legit, test_legit = legit.randomSplit([0.8, 0.2], seed=42)

train_df = train_fraud.union(train_legit)
test_df = test_fraud.union(test_legit)


# Columns definition


categorical_cols = [
    "merchant_category",
    "device_type",
    "channel",
    "location_city_grouped"
]

numeric_cols = [
    "log_amount",
    "amount_balance_ratio",
    "transaction_hour",
    "transaction_dayofweek",
    "merchant_fraud_rate",
    "customer_avg_transaction_amount"
]

# Now I apply StringIndexer for categorical columns

indexers = [
    StringIndexer(inputCol=col_name, outputCol=col_name + "_index", handleInvalid="keep")
    for col_name in categorical_cols
]

# Now columns are combined aplying VectorAssembler

feature_cols = numeric_cols + [c + "_index" for c in categorical_cols]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Creation of pipeline

pipeline = Pipeline(stages=indexers + [assembler])

pipeline_model = pipeline.fit(train_df)

train_final = pipeline_model.transform(train_df).select(
    "transaction_id",
    "customer_id",
    "features",
    "is_fraud"
)

test_final = pipeline_model.transform(test_df).select(
    "transaction_id",
    "customer_id",
    "features",
    "is_fraud"
)

# Saving 

train_final.write.format("delta")\
    .mode("overwrite") \
    .saveAsTable("ml_layer.transaction_train_features")

test_final.write.format("delta")\
    .mode("overwrite") \
    .saveAsTable("ml_layer.transaction_test_features")